In [1]:
# ----------------- Step 0: Install Required Packages -----------------
!pip install -q pandas pyarrow sentence-transformers faiss-cpu chromadb openai

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 24.1.2 -> 25.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
import pandas as pd

df = pd.read_parquet(r"C:\Users\manjo\Downloads\wiki_hybrid_chunks_300 (1).parquet")
print("Available columns:\n", df.columns)

Available columns:
 Index(['group_id', 'section', 'chunk_id', 'chunk_text'], dtype='object')


In [5]:
# Use the correct text column
df['text'] = df['chunk_text'].astype(str)
texts = df['text'].tolist()

In [6]:
# -------------------- Step 2: Generate Embeddings --------------------
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = model.encode(texts, show_progress_bar=True)


Batches:   0%|          | 0/314 [00:00<?, ?it/s]

In [7]:
# -------------------- Step 3: Save Embeddings to New Parquet --------------------
import numpy as np

embedding_df = pd.DataFrame(embeddings, columns=[f"emb_{i}" for i in range(len(embeddings[0]))])
df_with_embeddings = pd.concat([df.reset_index(drop=True), embedding_df], axis=1)
df_with_embeddings.to_parquet("wiki_chunks_with_embeddings.parquet")
print("✅ Saved: wiki_chunks_with_embeddings.parquet")

✅ Saved: wiki_chunks_with_embeddings.parquet


In [8]:
# -------------------- Step 4A: Store in FAISS --------------------
import faiss

embedding_array = np.array(embeddings).astype("float32")
faiss_index = faiss.IndexFlatL2(embedding_array.shape[1])
faiss_index.add(embedding_array)
faiss.write_index(faiss_index, "wiki_chunks_faiss.index")
print("✅ Saved FAISS index")

✅ Saved FAISS index


In [10]:
# -------------------- Step 5: Retrieve Relevant Chunks --------------------
query = "What is the use of reinforcement learning in chatbots?"
query_embedding = model.encode([query]).astype("float32")

D, I = faiss_index.search(query_embedding, k=3)
retrieved_chunks = [texts[i] for i in I[0]]
retrieved_context = " ".join(retrieved_chunks)

In [ ]:
# Generate Answer Using Local LLM

In [12]:
pip install transformers accelerate torch

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.1.2 -> 25.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
#load a local LLM (Lightweight Example)

In [13]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, device_map="auto", torch_dtype=torch.float16)

tokenizer_config.json:   0%|          | 0.00/1.29k [00:00<?, ?B/s]

C:\Users\manjo\AppData\Roaming\Python\Python312\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\manjo\.cache\huggingface\hub\models--TinyLlama--TinyLlama-1.1B-Chat-v1.0. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [ ]:
#Build prompt from Retrieved Chunks

In [14]:
def build_prompt(context, query):
    return f"""You are an intelligent assistant. Use the following context to answer the question.

Context:
{context}

Question: {query}
Answer:"""

In [ ]:
#Generate Answer from LLM

In [17]:
def generate_answer(prompt):
    inputs = tokenizer(prompt, return_tensors="pt").to("cpu")
    output = model.generate(**inputs, max_new_tokens=256, temperature=0.7)
    return tokenizer.decode(output[0], skip_special_tokens=True)

In [ ]:
#combine eveything

In [18]:
query = "What is the use of reinforcement learning in chatbots?"

# You already have this from FAISS
retrieved_context = " ".join(retrieved_chunks)

# Build prompt and get answer
prompt = build_prompt(retrieved_context, query)
answer = generate_answer(prompt)

print("Answer:\n", answer)

C:\Users\manjo\AppData\Roaming\Python\Python312\site-packages\transformers\generation\configuration_utils.py:631: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(


Answer:
 You are an intelligent assistant. Use the following context to answer the question.

Context:
about how they spend or invest money. a player s strategy usually revolves around brass franchise games the business game terra mystica franchise games great western trail catan food chain magnate the game of life monopoly power grid ensuring they have enough resources to achieve a strong financial position. economic board games usually simulate a market in some way. these games are often also called economic simulation games. the term economic board game is often used interchangeably with resource management board game. terraforming mars through the ages twilight imperium educational educational board games are those designed to teach new ideas, concepts, topics, and or understanding, while playing. the board game s learning is based ona particular theme. while educational games exist for different age groups, they are usually designed for children. the magic labyrinth brain quest ca